# Модель предсказания возрастной группы посетителей сайтов

## Содержание
1. <a href="#intro">Введение</a>
1. <a href="#preparation">Подотовка данных</a>
1. <a href="#eda">Исследовательский анализ данных</a>
1. <a href="#preprocessing">Предобработка датасета</a>
1. <a href="#baseline">Обучение и оценка базовой модели</a>
1. <a href="#feature-engineering">Создание и отбор признаков</a>
1. <a href="#hyperparameters">Подбор гиперпараметров</a>
1. <a href="#save-model">Подготовка артефактов модели для внедрения</a>
1. <a href="#final-report">Выводы исследования</a>

## Введение

### Бизнес-контекст

Компания Йети, владелет рекламной сети РСЙ, для улучшения маркетинговых рекламных кампаний решила использовать дополнительные сведения о возрасте посетителей сайтов. Компания владеет большим объемом информации о сессиях пользователей на различных рекламных площадках, например, о категориях сайтов, о глубине просмотра и так далее.

Требуется построить модель машинного обучения, которая бы предсказывала по перечисленным признакам возрастную группу пользователя. Данный признак в совокупности с другой информацией о пользователе позволит более эффективно участвовать в рекламных аукционах и подбирать для пользователей подходящие рекламные объявления.

Кроме того, в некоторых странах, за показ несоответствующей рекламы для определенных групп пользователей предусмотрены юридические последствия, поэтому важно определять.

### Цель исследования

Построить модель машинного обучения, которая по различным признакам пользователя будет предсказывать его возрастную группу в форме одной из категорий с диапазонами лет.

### Постановка задачи машинного обучения

В предоставленных данных имеется целевая переменная - возрастная группа пользователя. Мы имеем дело с задачей обучение с учителем. Возрастные группы подразделяются на:
- 0: младше 18;
- 1: 18-25 лет;
- 2: 26-40 лет;
- 3: 41-55 лет;
- 4: 56+ лет.

Целевая переменная представляет несколько допустимых значений (возрастных диапазонов), таким образом необходимо решить задачу многоклассовой классификации.

В качестве модели будем использовать несколько вариантов:
- Логистическая регрессия, как базовый алгоритм
- Метод опорных векторов с различными ядрами для улавливания нелинейных зависимостей признаков с целевой переменной.

**Метрики качества**

В роли основной метрики качества используем F1-меру с макро ксредненим для того, чтобы оценить модель даже на узко представленных возрастных группах. Вспомогательные метрики $Precision(k)_{macro}$, $Recall(k)_{macro}$ для полного понимани сильных и слабых сторон модели в целях бизнес-анализа.

Значение F1-меры лучшей модели должно быть не меньше 0.75 на тесте для успешного внедрения модели в производственную эксплуатацию.

## Подготовка среды и библиотек

Установим зависимости:

In [6]:
from pathlib import Path

requirements_file = Path('requirements.txt')
requirements = [
    'phik==0.12.5',
    'joblib==1.5.3',
    'scikit-learn==1.6.1',
    'seaborn==0.13.2',
]
if not requirements_file.exists():
    with open(requirements_file, 'w') as f:
        f.write('\n'.join(requirements))
        print(f'{requirements_file} created')
else:
    print(f'{requirements_file} exists')

print('Installing requirements...')
!pip install -r requirements.txt
print('Requirements have been installed!')


requirements.txt created
Installing requirements...
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Requirements have been installed!


Импортируем необходимые классы и функции:

In [7]:
import os
import requests
from datetime import datetime
from time import time
import joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer

from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.metrics import classification_report
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import chi2, SelectKBest
from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.calibration import CalibratedClassifierCV

from scipy.stats import uniform, loguniform
from phik import phik_matrix


Зафиксируем RANDOM_STATE для воспроизводимости результатов и настроим отображение ячеек в ноутбуке:

In [8]:
RANDOM_STATE = 153
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', None) # выводить все колонки
pd.set_option('display.max_colwidth', 500) # выводить больше символов в ячейке


Загрузим датасеты для последующего объединения:

In [10]:
def load_dataset(
    dataset_url,
    local_file,
    local_path='datasets',
    sep=',',
    decimal='.'
):
    local_dataset_file = f'{local_path}/{local_file}'
    remote_dataset_url = dataset_url
    def read_dataset_csv():
        return pd.read_csv(local_dataset_file, sep=sep, decimal=decimal)

    try:
        df = read_dataset_csv()
        print(f'Датасет успешно загружен из {local_dataset_file}')
    except FileNotFoundError:
        os.makedirs(local_path, exist_ok=True)
        print(f'Файл не найден. Загружаем файл в {local_dataset_file} из {remote_dataset_url}')
        response = requests.get(remote_dataset_url)
        if response.status_code == 200:
            with open(local_dataset_file, 'wb') as f:
                f.write(response.content)
            print(f'Файл с датасетом успешно загружен в {local_dataset_file}')
            df = read_dataset_csv()
        else:
            raise NetworkError(f'Ошибка при загрузке файла: {response.status_code}')

    print(f'Размер загруженного датасета: {df.shape[0]} строк, {df.shape[1]} столбцов', )
    return df

df_users = load_dataset(
    dataset_url='https://code.s3.yandex.net/datasets/ds_s13_users.csv',
    local_file='ds_s13_users.csv'
)
df_visits = load_dataset(
    dataset_url='https://code.s3.yandex.net/datasets/ds_s13_visits.csv',
    local_file='ds_s13_visits.csv'
)
df_ads_activity = load_dataset(
    dataset_url='https://code.s3.yandex.net/datasets/ads_activity.csv',
    local_file='ads_activity.csv'
)
df_surf_depth = load_dataset(
    dataset_url='https://code.s3.yandex.net/datasets/surf_depth.csv',
    local_file='surf_depth.csv'
)
df_primary_device = load_dataset(
    dataset_url='https://code.s3.yandex.net/datasets/primary_device.csv',
    local_file='primary_device.csv'
)
df_cloud_usage = load_dataset(
    dataset_url='https://code.s3.yandex.net/datasets/cloud_usage.csv',
    local_file='cloud_usage.csv'
)

Файл не найден. Загружаем файл в datasets/ds_s13_users.csv из https://code.s3.yandex.net/datasets/ds_s13_users.csv
Файл с датасетом успешно загружен в datasets/ds_s13_users.csv
Размер загруженного датасета: 5913 строк, 2 столбцов
Файл не найден. Загружаем файл в datasets/ds_s13_visits.csv из https://code.s3.yandex.net/datasets/ds_s13_visits.csv
Файл с датасетом успешно загружен в datasets/ds_s13_visits.csv
Размер загруженного датасета: 1065745 строк, 5 столбцов
Файл не найден. Загружаем файл в datasets/ads_activity.csv из https://code.s3.yandex.net/datasets/ads_activity.csv
Файл с датасетом успешно загружен в datasets/ads_activity.csv
Размер загруженного датасета: 5826 строк, 2 столбцов
Файл не найден. Загружаем файл в datasets/surf_depth.csv из https://code.s3.yandex.net/datasets/surf_depth.csv
Файл с датасетом успешно загружен в datasets/surf_depth.csv
Размер загруженного датасета: 5715 строк, 2 столбцов
Файл не найден. Загружаем файл в datasets/primary_device.csv из https://code.s3.

## Исследовательский анализ данных

## Предобработка данных

## Обучение и оценка базовой модели

## Создание и отбор признаков

## Подбор гиперпараметров моделей

## Подготовка артефактов модели для внедрения

## Выводы о результатах работы